anomaly_context.json
        │
        ▼
Load JSON
        │
        ▼
Create Prompt
        │
        ▼
Groq API
        │
        ▼
Root Cause Analysis
        │
        ▼
Recommendations
        │
        ▼
Save Report

# AI-Powered Root Cause Analysis using Groq

This notebook performs automated Root Cause Analysis (RCA) on anomalous logs using a Large Language Model served through the Groq API.

For each detected anomaly, the surrounding context is analysed to determine:

- Probable Root Cause
- Severity
- Impact
- Recommended Fixes
- Confidence Score

The generated report can assist Site Reliability Engineers (SREs) in quickly identifying and resolving production issues.

In [1]:
!pip3 install groq
!pip3 install pandas 



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: /Library/Frameworks/Python.framework/Versions/3.14/bin/python3.14 -m pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: /Library/Frameworks/Python.framework/Versions/3.14/bin/python3.14 -m pip install --upgrade pip


In [2]:
pip install python-dotenv


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import json
from pathlib import Path

import pandas as pd

from groq import Groq

# Loading Context 

In [4]:
with open("../data/context/anomaly_context.json","r") as f:
    contexts = json.load(f)

print(f"Loaded {len(contexts)} anomaly contexts.")

Loaded 248 anomaly contexts.


In [5]:
from dotenv import load_dotenv
from groq import Groq
import os

load_dotenv(override=True)

client = Groq(
    api_key=os.getenv("GROQ_API_KEY")
)

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "user", "content": "Say SUCCESS"}
    ]
)

print(response.choices[0].message.content)

SUCCESS!


In [6]:
import json

def build_prompt(context):

    return f"""
You are an expert Site Reliability Engineer (SRE), DevOps Engineer, and Production Support Specialist.

Analyze the following anomalous log and perform a comprehensive Root Cause Analysis (RCA).

For every issue:
- Identify the most likely root cause.
- Explain why it happened.
- Assess the severity and business impact.
- Suggest immediate troubleshooting steps.
- Suggest permanent fixes.
- Recommend preventive measures to avoid similar incidents in the future.
- Estimate your confidence level.

Return ONLY valid JSON in the following format:

{{
    "RootCause": "...",
    "Reason": "...",
    "Severity": "Critical | High | Medium | Low",
    "Impact": "...",

    "ImmediateFixes": [
        "...",
        "...",
        "..."
    ],

    "PermanentFixes": [
        "...",
        "...",
        "..."
    ],

    "PreventiveMeasures": [
        "...",
        "...",
        "..."
    ],

    "Recommendations": [
        "...",
        "...",
        "..."
    ],

    "Confidence": "95%"
}}

Current Log

{context["CurrentLog"]}

Component

{context["Component"]}

Severity

{context["Severity"]}

Contains Error

{context["ContainsError"]}

Authentication Success

{context["AuthenticationSuccess"]}

Anomaly Score

{context["AnomalyScore"]}

Nearby Logs

{json.dumps(context["NearbyLogs"], indent=4)}

Provide concise, technically accurate, and actionable recommendations suitable for production environments.
"""

In [ ]:
import time

TOP_K = 5

rca_results = []

for i, context in enumerate(contexts[:TOP_K]):

    print(f"Processing {i+1}/{TOP_K}")

    prompt = build_prompt(context)

    response = client.chat.completions.create(

        model="llama-3.3-70b-versatile",

        temperature=0,

        messages=[
            {
                "role":"user",
                "content":prompt
            }
        ]

    )

    rca_results.append({

        "LogIndex": context["LogIndex"],

        "CurrentLog": context["CurrentLog"],

        "Component": context["Component"],

        "OriginalSeverity": context["Severity"],

        "ContainsError": context["ContainsError"],

        "AuthenticationSuccess": context["AuthenticationSuccess"],

        "AnomalyScore": context["AnomalyScore"],

        "NearbyLogs": context["NearbyLogs"],

        "Response": response.choices[0].message.content

    })

    time.sleep(1)

Processing 1/5


RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kv7rvnkzeh28t2c7xv2b509j` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99933, Requested 2014. Please try again in 28m2.208s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

## Parse JSON Responses

The responses returned by the language model are cleaned and converted into structured Python dictionaries.

Any malformed responses are skipped gracefully.

In [ ]:
import json
import re

parsed_results = []

for item in rca_results:

    try:

        response = item["Response"]

        response = response.replace("```json","")
        response = response.replace("```","")
        response = response.strip()

        match = re.search(r"\{.*\}", response, re.DOTALL)

        if match:

            result = json.loads(match.group())

            result["LogIndex"] = item["LogIndex"]
            result["CurrentLog"] = item["CurrentLog"]
            result["Component"] = item["Component"]
            result["OriginalSeverity"] = item["OriginalSeverity"]
            result["ContainsError"] = item["ContainsError"]
            result["AuthenticationSuccess"] = item["AuthenticationSuccess"]
            result["AnomalyScore"] = item["AnomalyScore"]
            result["NearbyLogs"] = item["NearbyLogs"]

            parsed_results.append(result)

        else:

            print(f"No JSON found for Log {item['LogIndex']}")

    except Exception as e:

        print(f"Skipped Log {item['LogIndex']} : {e}")

## Create Structured RCA Report

The parsed JSON responses are converted into a Pandas DataFrame for further analysis and visualization.

In [ ]:
rca_df = pd.DataFrame(parsed_results)

rca_df.head()

,RootCause,Reason,Severity,Impact,ImmediateFixes,PermanentFixes,PreventiveMeasures,Recommendations,Confidence,LogIndex
0,High TPS (Transactions Per Second) causing sys...,The system is experiencing a high volume of tr...,High,The high TPS is causing the system to become o...,[Implement rate limiting to reduce the number ...,[Optimize system configuration and architectur...,[Monitor system performance and TPS regularly ...,[Investigate the root cause of the high TPS an...,90%,4526
1,High TPS (Transactions Per Second) causing sys...,The system is experiencing a high volume of tr...,High,"The high TPS may cause delays, errors, or even...",[Monitor system performance and adjust TPS lim...,[Optimize system configuration and code to imp...,[Regularly monitor system performance and adju...,[Analyze system logs to identify patterns and ...,90%,1101
2,High TPS (Transactions Per Second) causing sys...,The system is experiencing a high volume of tr...,High,"Potential delay or loss of SMS messages, impac...",[Monitor system performance and adjust TPS lim...,[Optimize system configuration and code to imp...,[Implement monitoring and alerting systems to ...,[Conduct a thorough review of system architect...,90%,4944
3,High TPS (Transactions Per Second) causing sys...,The system is experiencing a high volume of tr...,High,"Potential system overload, performance degrada...","[Monitor system resources (CPU, memory, disk u...",[Upgrade system hardware or infrastructure to ...,[Regularly monitor system performance and adju...,[Conduct a thorough system audit to identify p...,90%,3605
4,Insufficient balance for subscription to EXPLO...,The user's account balance is not sufficient t...,Medium,The user is unable to subscribe to the EXPLORE...,[Verify the user's account balance and ensure ...,[Implement a check for sufficient balance befo...,[Regularly review and update the EXPLORE servi...,[Consider implementing a more robust balance c...,90%,2237


In [ ]:
from pathlib import Path

Path("../outputs").mkdir(exist_ok=True)

rca_df.to_csv(
    "../outputs/root_cause_analysis.csv",
    index=False
)

print("Root Cause Analysis saved successfully.")

Root Cause Analysis saved successfully.


In [ ]:
rca_df = pd.DataFrame(parsed_results)

rca_df.head()

""


In [ ]:
from pathlib import Path

Path("../outputs").mkdir(exist_ok=True)

rca_df.to_csv(
    "../outputs/root_cause_analysis.csv",
    index=False
)

print("Root Cause Analysis saved successfully.")

Root Cause Analysis saved successfully.


In [ ]:
print("Total Anomalies Analysed :", len(rca_df))

print("\nSeverity Distribution")
display(rca_df["Severity"].value_counts())

print("\nMost Common Root Causes")
display(rca_df["RootCause"].value_counts())

print("\nConfidence Distribution")
display(rca_df["Confidence"].value_counts())

Total Anomalies Analysed : 20

Severity Distribution


Severity
High      8
Medium    8
Low       4
Name: count, dtype: int64


Most Common Root Causes


RootCause
Insufficient balance for subscription to EXPLORE service                  11
High TPS (Transactions Per Second) causing system overload                 5
High Transaction Per Second (TPS) rate exceeding system capacity           1
Invalid Destination Address                                                1
High Transaction Per Second (TPS) rate exceeding the system's capacity     1
High Transaction Per Second (TPS) rate                                     1
Name: count, dtype: int64


Confidence Distribution


Confidence
90%    20
Name: count, dtype: int64

# Each single anomaly report 

In [ ]:
# from IPython.display import Markdown, display

# for _, row in rca_df.iterrows():

#     display(Markdown(f"""
# # 🚨 Anomaly #{row['LogIndex']}

# ### 🔍 Root Cause
# {row['RootCause']}

# ### 📖 Reason
# {row['Reason']}

# ### ⚠️ Severity
# **{row['Severity']}**

# ### 💥 Impact
# {row['Impact']}

# ### 🚑 Immediate Fixes
# """))

#     for fix in row["ImmediateFixes"]:
#         print(f"• {fix}")

#     display(Markdown("### 🔧 Permanent Fixes"))

#     for fix in row["PermanentFixes"]:
#         print(f"• {fix}")

#     display(Markdown("### 🛡 Preventive Measures"))

#     for fix in row["PreventiveMeasures"]:
#         print(f"• {fix}")

#     display(Markdown("### 💡 Recommendations"))

#     for rec in row["Recommendations"]:
#         print(f"• {rec}")

#     display(Markdown(f"""
# ### 🎯 Confidence
# **{row['Confidence']}**

# ---
# """))

# Function to find any anomaly report 

In [ ]:
def show_anomaly(log_index):

    row = rca_df[rca_df["LogIndex"] == log_index].iloc[0]

    print("=" * 80)
    print(f"Anomaly ID : {row['LogIndex']}")
    print(f"Current Log : {row['CurrentLog']}")
    print(f"Component : {row['Component']}")
    print(f"Anomaly Score : {row['AnomalyScore']}")
    print(f"Root Cause : {row['RootCause']}")
    print(f"Reason : {row['Reason']}")
    print(f"Severity : {row['Severity']}")
    print(f"Impact : {row['Impact']}")

    print("\nImmediate Fixes")
    for fix in row["ImmediateFixes"]:
        print("•", fix)

    print("\nPermanent Fixes")
    for fix in row["PermanentFixes"]:
        print("•", fix)

    print("\nPreventive Measures")
    for fix in row["PreventiveMeasures"]:
        print("•", fix)

    print("\nRecommendations")
    for rec in row["Recommendations"]:
        print("•", rec)

    print(f"\nConfidence : {row['Confidence']}")
    print("=" * 80)

In [ ]:
print(rca_df.columns.tolist())

['RootCause', 'Reason', 'Severity', 'Impact', 'ImmediateFixes', 'PermanentFixes', 'PreventiveMeasures', 'Recommendations', 'Confidence', 'LogIndex']


In [ ]:
show_anomaly(4526)

Anomaly ID : 4526


KeyError: 'CurrentLog'